# Mimicking RouteLLM

### Generate embeddings

In [24]:
import os
import json
import requests
import numpy as np

results_path = '../AutoFL/results/d4j_autofl_eol_1/llama3'
initial_prompts_path = 'route_data/initial_prompts.json'
endpoint = 'http://localhost:11434/api/embeddings'
embeddings_path = 'route_data/embeddings.json'

In [ ]:
initial_prompt_dict = dict()
for file in sorted(os.listdir(results_path)):
    with open(os.path.join(results_path, file)) as f:
        msgs = json.load(f)['messages']
        bug_id = file[4:-5]
        initial_prompt = msgs[0]['content'] + msgs[1]['content']
        initial_prompt_dict[bug_id] = initial_prompt

with open(initial_prompts_path, 'w') as f:
    json.dump(initial_prompt_dict, f, indent=4)

In [ ]:
def _query_model(payload):
    for _ in range(5):
        try:
            json_payload = json.dumps(payload)
            headers = {'Content-Type': 'application/json'}
            response = json.loads(requests.post(endpoint, data=json_payload, headers=headers).text)
            return response['embedding']
        except Exception as e:
            save_err = e
            if "The server had an error processing your request." in str(e):
                time.sleep(1)
            else:
                break
    raise save_err


def get_LLM_response(initial_prompt):
    payload = {
        'model': 'nomic-embed-text',
        'prompt': initial_prompt,
        'stream': False
    }
    return _query_model(payload)

if __name__ == "__main__":
    embeddings_dict = dict()
    with open(initial_prompts_path) as f:
        data = json.load(f)
        
    for bug_id in data:
        initial_prompt = data[bug_id]
        embedding = get_LLM_response(initial_prompt)
        embeddings_dict[bug_id] = embedding
    
    with open(embeddings_path, 'w') as f:
        json.dump(embeddings_dict, f)

### Format vs. results

In [43]:
gpt_result_path = '../AutoFL/combined_fl_results/d4j_gpt4o_results_R10_full.json'
llama_result_path = '../AutoFL/combined_fl_results/d4j_eol_llama3_R10.json'
vs_data_path = 'route_data/vs.json'
filtered_embeddings_path = 'route_data/embeddings.npy'

with open(gpt_result_path) as f:
    gpt_result = json.load(f)['buggy_methods']

with open(llama_result_path) as f:
    llama_result = json.load(f)['buggy_methods']

In [44]:
def extract_min_ranks(result, bug_list):
    min_ranks = list() 
    for bug_id in bug_list:
        buggy_methods = result[bug_id]
        min_ranks.append(0 if not buggy_methods else min(map(lambda x: x['autofl_rank'], buggy_methods.values())))
    
    return min_ranks    

In [45]:
sorted_bug_list = sorted(gpt_result.keys())

gpt_ranks = extract_min_ranks(gpt_result, sorted_bug_list)
llama_ranks = extract_min_ranks(llama_result, sorted_bug_list)
vs_result = list(map(lambda x: x[0] < x[1], zip(gpt_ranks, llama_ranks)))

In [46]:
vs_data = list()
for i, bug_id in enumerate(sorted_bug_list):
    vs_data.append({
        "model_a": 'gpt-4o',
        "model_b": 'llama3-8b',
        "idx": i,
        "winner": "model_a" if vs_result[i] else "model_b"
    })

with open(vs_data_path, 'w') as f:
    json.dump(vs_data, f, indent=4)

In [ ]:
with open(embeddings_path) as f:
    embeddings = json.load(f)
filtered_embeddings = np.array([embeddings[bug_id] for bug_id in sorted_bug_list])
np.save(filtered_embeddings_path, filtered_embeddings)

### How to use?

In [3]:
ck_path = 'routers/matrix_factorization/best_checkpoint.pt'

from routers.routers import MatrixFactorizationRouter
router = MatrixFactorizationRouter(ck_path, strong_model='gpt-4o', weak_model='llama3-8b')

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'routers/matrix_factorization/best_checkpoint.pt'. Use `repo_type` argument if needed.